In [ ]:
import warnings
import re
warnings.filterwarnings('ignore')
import os
# Read data
import pandas as pd
import numpy as np
import polars as pl
import polars.selectors as cs
pl.Config.set_tbl_rows(20)  # show up to 20 rows

# Visualization
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set_style("darkgrid")  # darkgrid

# Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Import CuPy and cuML
import cupy as cp
from cuml.cluster import KMeans as cuMLKMeans
from cuml.ensemble import RandomForestClassifier as cuMLRandomForestClassifier

In [ ]:
accepted_df = pl.read_parquet("../data/processed_data/processed_accepted_df.parquet")
X = accepted_df.drop("target")
y = accepted_df.select("target")

FileNotFoundError: The system cannot find the file specified. (os error 2): accepted_2007_to_2018Q4.parquet

## Classification Models for Default Prediction

**What to do:**

- Train 3 classifiers on the default prediction task: Logistic Regression, Random Forest, XGBoost
- Handle class imbalance using class weighting (`class_weight='balanced'`) or SMOTE
- Tune key hyperparameters for RF and XGBoost (n_estimators, max_depth, learning_rate) using cross-validation
- Evaluate on test set: F1, Precision, Recall, AUC-ROC for the default (minority) class
- Extract and plot feature importances from XGBoost - which features drive default prediction most?
- Convert best classifier into a static lending policy: approve if P(default) < threshold, assign rate tier by risk score - then run through `LendingEnv` to get profit/NPL metrics for comparison

**Input needed:** Preprocessed feature matrix + train/test split from EDA - ✅ already available

**Deliverables:**

- Classification metrics table (LR vs RF vs XGBoost): F1, Precision, Recall, AUC
- Feature importance bar chart (top 10 features from XGBoost)
- Static policy metrics from `LendingEnv`: profit, NPL, approval rate (same format as RL methods)

**Deadline:** `[TBD]`

In [ ]:
# convert 'term' → integer
X = X.with_columns(
    pl.col("term")
    .str.extract(r"(\d+)", 1)
    .cast(pl.Int64)
)

# convert 'emp_length' → integer
X = X.with_columns(
    pl.when(pl.col("emp_length") == "< 1 year").then(0)
    .when(pl.col("emp_length") == "10+ years").then(10)
    .otherwise(
        pl.col("emp_length")
        .str.extract(r"(\d+)", 1)
        .cast(pl.Int64)
    )
    .alias("emp_length")
)

# encode categorical variables (Polars way)
X = X.with_columns([
    pl.col("grade").cast(pl.Categorical).to_physical().alias("grade"),
    pl.col("purpose").cast(pl.Categorical).to_physical().alias("purpose")
])

# drop unwanted columns
X = X.drop([
    "sub_grade",
    "home_ownership",
    "earliest_cr_line",
    "verification_status"
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.3)

In [ ]:
"""KFold cross validation with split preservation of sample percentage for each class"""
def stratified_kfold_split(model, X, y, n_fold):
    X, y = X.values, y.values
    skf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=42)
    precision_arr = []
    for train_idx, test_idx in skf.split(X, y):
        X_train_fold, X_test_fold, y_train_fold, y_test_fold = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
        model.fit(X_train_fold, y_train_fold)
        preds = model.predict(X_test_fold)
        precision_arr.append(precision_score(preds, y_test))

    return np.array(precision_arr).mean()

"""Black box function"""
def bo_rf_func(n_estimators: int, max_depth: int,
               max_samples: float, max_features: float):

    # create model instance
    params = {"n_estimators": int(n_estimators),
              "max_depth": int(max_depth),
              "max_samples": max_samples,
              "max_features": max_features}
    rf = RandomForestClassifier(n_estimators=params["n_estimators"],
                                max_depth=params["max_depth"],
                                max_samples=params["max_samples"],
                                max_features=params["max_features"])
    score = stratified_kfold_split(rf, X, y, 5)
    return score

def bo_xgb_func(n_estimators: int, max_depth: int,
                    subsample: float, objective: str, eta: float):
    # create model instance
    params = {"n_estimators": int(n_estimators),
              "max_depth": int(max_depth),
              "subsample": subsample,
              "eta": eta}
    xgboost = XGBClassifier(n_estimators=params["n_estimators"],
                            max_depth=params["max_depth"],
                            subsample=params["subsample"],
                            eta=params["eta"], objective="binary:hinge")
    score = stratified_kfold_split(xgboost, X, y, n_fold=5)
    return score

### 1. Random Forest

In [ ]:
X.head()

loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,emp_length,fico_score_avg,last_pymnt_amnt,tot_cur_bal,pub_rec_bankruptcies,tax_liens,acc_now_delinq,mort_acc,bc_util,percent_bc_gt_75,num_rev_accts,num_tl_90g_dpd_24m
f64,i64,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3600.0,36,13.99,1,7,55000.0,5.91,0.0,1.0,7.0,0.0,2765.0,29.7,10,677.0,122.67,144904.0,0.0,0.0,0.0,1.0,37.2,0.0,9.0,0.0
24700.0,36,11.99,1,8,65000.0,16.06,1.0,4.0,22.0,0.0,21470.0,19.2,10,717.0,926.35,204396.0,0.0,0.0,0.0,4.0,27.1,7.7,27.0,0.0
20000.0,60,10.78,3,9,63000.0,10.78,0.0,0.0,6.0,0.0,7869.0,56.2,10,697.0,15813.3,189699.0,0.0,0.0,0.0,5.0,55.9,50.0,7.0,0.0
10400.0,60,22.45,5,10,104433.0,25.37,1.0,3.0,12.0,0.0,21929.0,64.5,3,697.0,10128.96,331730.0,0.0,0.0,0.0,6.0,77.5,60.0,19.0,0.0
11950.0,36,13.44,1,7,34000.0,10.2,0.0,0.0,5.0,0.0,8822.0,68.4,4,692.0,7653.56,12798.0,0.0,0.0,0.0,0.0,91.0,100.0,4.0,0.0


In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_preds = rf.predict(X_test)
print("--- Metrics evaluation ---\n")
print(f"Precision score: {precision_score(y_preds, y_test):.3f}")
print(f"Recall score: {recall_score(y_preds, y_test):.3f}")
print(f"F1 score: {f1_score(y_preds, y_test):.3f}")
print(classification_report(y_preds, y_test))

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


--- Metrics evaluation ---

Precision score: 0.643
Recall score: 0.781
F1 score: 0.705
              precision    recall  f1-score   support

           0       0.96      0.91      0.93    337278
           1       0.64      0.78      0.71     66315

    accuracy                           0.89    403593
   macro avg       0.80      0.85      0.82    403593
weighted avg       0.90      0.89      0.90    403593



In [ ]:
rf_bounds = {"n_estimators": (100, 500),
             "max_depth": (10, 100),
             "max_samples": {0.5, 1.0},
             "max_features": {0.1, 1.0}}

rf_bo = BayesianOptimization(f=bo_rf_func, pbounds=rf_bounds, ramdom_state=42)
rf_results = rf_bo.maximize(n_iter=20, init_points=200)

TypeError: BayesianOptimization.__init__() got an unexpected keyword argument 'ramdom_state'

In [ ]:
rf_params = rf_bo.max['params']
rf_params['n_estimators']= int(rf_params['n_estimators'])
rf_params["max_depth"] = int(rf_params["max_depth"])
print(rf_params)

### 2. XGBoost

In [ ]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_preds = xgb.predict(X_test)
print("--- Metrics evaluation ---\n")
print(f"Precision score: {precision_score(y_preds, y_test):.3f}")
print(f"Recall score: {recall_score(y_preds, y_test):.3f}")
print(f"F1 score: {f1_score(y_preds, y_test):.3f}")
print(classification_report(y_preds, y_test))

In [ ]:
xgb_bounds = {"n_estimators": (100, 500),
             "max_depth": (10, 100),
             "subsample": {0.5, 1.0},
              "eta": (0.001, 1)}

xgb_bo = BayesianOptimization(f=bo_xgb_func, pbounds=xgb_bounds, random_state=42)
xgb_results = xgb_bo.maximize(n_iter=20, init_points=200)

In [ ]:
xgb_params = xgb_bo.max['params']
xgb_params['n_estimators']= int(xgb_params['n_estimators'])
xgb_params["max_depth"] = int(xgb_params["max_depth"])
print(xgb_params)

### K-Fold Validation for Best Params

In [ ]:
X_pd = X.to_pandas()
y_pd = y.to_pandas()

In [ ]:
# store SHAP importances
rf_shap_importance = []
xgb_shap_importance = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
feature_names = X_pd.columns

# Random Forest + SHAP
for train_idx, test_idx in cv.split(X_pd, y_pd):
    X_train, X_test = X_pd.iloc[train_idx], X_pd.iloc[test_idx]
    y_train, y_test = y_pd.iloc[train_idx], y_pd.iloc[test_idx]

    model = RandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    # classification → take class 1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    importance = np.abs(shap_values).mean(axis=0)
    rf_shap_importance.append(importance)

# XGBoost + SHAP
for train_idx, test_idx in cv.split(X_pd, y_pd):
    X_train, X_test = X_pd.iloc[train_idx], X_pd.iloc[test_idx]
    y_train, y_test = y_pd.iloc[train_idx], y_pd.iloc[test_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    importance = np.abs(shap_values).mean(axis=0)
    xgb_shap_importance.append(importance)

In [ ]:
def plot_importance(importance, feature_names, title):
    idx = np.argsort(importance)

    plt.figure()
    plt.barh(range(len(idx)), importance[idx])
    plt.yticks(range(len(idx)), feature_names[idx])
    plt.title(title)
    plt.xlabel("Mean |SHAP value|")
    plt.show()

In [ ]:
rf_importance_mean = np.mean(rf_shap_importance, axis=0)
xgb_importance_mean = np.mean(xgb_shap_importance, axis=0)

# Plot
plot_importance(rf_importance_mean, feature_names, "RF SHAP (CV averaged)")
plot_importance(xgb_importance_mean, feature_names, "XGB SHAP (CV averaged)")

In [ ]:
df_compare = pd.DataFrame({
    "feature": feature_names,
    "RF": rf_importance_mean,
    "XGB": xgb_importance_mean
}).sort_values("XGB", ascending=False)

print(df_compare.head(10))

In [ ]:
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# result = {"model": [], "accuracy": [], "precision": [], "recall": [], "f1": []}

# X_pd = X.to_pandas()
# y_pd = y.to_pandas()

# # Random Forest + SHAP
# for train_idx, test_idxin cv.split(X, y):
#     X_train, X_test = X_pd.iloc[train_idx], X_pd.iloc[test_idx]
#     y_train, y_test = y_pd.iloc[train_idx], y_pd.iloc[test_idx]
#     best_rf = RandomForestClassifier(**rf_params)
#     best_rf.fit(X.iloc[train_idx], y.iloc[train_idx])
#     rf_preds = best_rf.predict(X.iloc[test_idx])
#     # Update result
#     result["model"].append("Random Forest")
#     result["accuracy"].append(accuracy_score(rf_preds, y.iloc[test_idx]))
#     result["precision"].append(precision_score(rf_preds, y.iloc[test_idx]))
#     result["recall"].append(recall_score(rf_preds, y.iloc[test_idx]))
#     result["f1"].append(f1_score(rf_preds, y.iloc[test_idx]))

# # XGBoost model
# best_xgb = XGBClassifier(**xgb_params)
# for (train_idx, test_idx), i in zip(cv.split(X, y), range(5)):
#     best_xgb.fit(X.iloc[train_idx], y.iloc[train_idx])
#     xgb_preds = best_xgb.predict(X.iloc[test_idx])
#     # Update result
#     result["model"].append("XGBoost")
#     result["accuracy"].append(accuracy_score(xgb_preds, y.iloc[test_idx]))
#     result["precision"].append(precision_score(xgb_preds, y.iloc[test_idx]))
#     result["recall"].append(recall_score(xgb_preds, y.iloc[test_idx]))
#     result["f1"].append(f1_score(xgb_preds, y.iloc[test_idx]))


## Unified Evaluation - Final Comaprison

**What to do:**

- Collect metrics from all 4 methods (Tasks 3, 4, 5 + baseline ✅) into one results table
- Build profit-risk tradeoff chart: plot cumulative profit vs NPL ratio for all 4 methods on one figure
- Write the Results & Discussion section of the paper:
    - Does the Constrained DQN outperform the bandit baseline on NPL compliance?
    - Does the penalty (Method 2 vs 3) make a meaningful difference?
    - Does the RL approach outperform the static classifier policy in profit?
    - What do the findings mean for a real bank's compliance team?

**Input needed:** Metrics from Tasks 3, 4, 5 - ⚠️ must wait for all three

**Deliverables:**

- Unified results table (all 4 methods, all metrics)
- Profit-risk tradeoff plot
- Written Results & Discussion section (draft)

**Deadline:** `[TBD - last task, after all others]`